# EXTRA EXERCISE 2
The laser engraving process is under study. The objective of the work is to evaluate the influence of process parameters, i.e scanning speed v(mm/s), pulse energy E(mJ) and number of passes N, on the depth of removed material, d (μm). The data are saved in the “Engraving.csv” file.

Suggest a model for the data. Use a stepwise regression approach. 

In [ ]:
# Import the necessary libraries
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy import stats
import seaborn as sns

# Import the dataset
data = pd.read_csv('../Data/Engraving.csv')

# Inspect the dataset
data.head()

## Point 1. Suggest a model for the data.

In [ ]:
# Plot the data using pairplot in seaborn, Quality vs all other features
fig, ax = plt.subplots(1,3, figsize=(12, 4), sharey=True)
for i in range(3):
    ax[i].scatter(data.iloc[:, i],data['d'], alpha=0.8)
    ax[i].set_xlabel(data.columns[i])
    ax[i].set_ylabel('d')
plt.show()


A linear relationship between d and N is visible. No clear pattern for d vs E and d vs v.

In [ ]:
import statsmodels.api as sm
import qdatoolkit as qda

# Compute all potential interaction terms
data['v*N'] = data['v'] * data['N']
data['v*E'] = data['v'] * data['E']
data['N*E'] = data['N'] * data['E']
data['v*N*E'] = data['v'] * data['N'] * data['E']

# Create X and y for the new model
X = data[['v','N','E','v*N','v*E','N*E','v*N*E']]
X = sm.add_constant(X)  # Add a constant term to the predictor
y = data['d']

In [ ]:
# Create a StepwiseRegression object using the qda library
stepwise = qda.StepwiseRegression(add_constant = True, direction='forward')
# Fit the model
model = stepwise.fit(y, X)

In [ ]:
# Print the summary of the model
model = model.model_fit
qda.summary(model)

Note: Results may vary from Minitab solutions as Minitab by default requires a hierarchical model at every step. 

In [ ]:
#NORMALITY OF RESIDUALS
residuals = model.resid
fig, axs = plt.subplots(2, 2)
fig.suptitle('Residual Plots')

axs[0,0].set_title('Normal probability plot')
stats.probplot(residuals, dist="norm", plot=axs[0,0])

axs[0,1].set_title('Versus Fits')
axs[0,1].scatter(model.fittedvalues, residuals)

fig.subplots_adjust(hspace=0.5)

axs[1,0].set_title('Histogram')
axs[1,0].hist(residuals)

axs[1,1].set_title('Time series plot')
axs[1,1].plot(np.arange(1, len(residuals)+1), residuals, 'o-')
plt.show()

In [ ]:
_ = qda.Assumptions(residuals).normality()

Normality is verified. The model is adequate. 